In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score

In [2]:
DATA_DIR = Path("../data/raw")

train = pd.read_csv(DATA_DIR / "train.csv")

print(train.shape)
display(train.head())

(1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
target = "SalePrice"
id_col = "Id"

y = train[target]
X = train.drop(columns=[target, id_col])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1460, 79)
y shape: (1460,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1168, 79)
X_test: (292, 79)
y_train: (1168,)
y_test: (292,)


In [5]:
def target_summary(series):
    return pd.Series({
        "mean": series.mean(),
        "median": series.median(),
        "min": series.min(),
        "max": series.max(),
        "skew": series.skew()
    })

summary = pd.DataFrame({
    "full": target_summary(y),
    "train": target_summary(y_train),
    "test": target_summary(y_test)
})

display(summary)

,full,train,test
mean,180921.195890,181441.541952,178839.811644
median,163000.000000,165000.000000,154150.000000
min,34900.000000,34900.000000,35311.000000
max,755000.000000,745000.000000,755000.000000
skew,1.882876,1.743129,2.264395


In [6]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numeric features: 36
['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']

Categorical features: 43
['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'Garage

C:\Users\Александр\AppData\Local\Temp\ipykernel_26996\3799334903.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()


In [7]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [8]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5

scoring = {
    "mae": make_scorer(mean_absolute_error, greater_is_better=False),
    "rmse": make_scorer(rmse, greater_is_better=False),
    "r2": make_scorer(r2_score)
}

In [9]:
models = {
    "DummyRegressor_mean": DummyRegressor(strategy="mean"),
    "DummyRegressor_median": DummyRegressor(strategy="median"),
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "DecisionTreeRegressor_max_depth_3": DecisionTreeRegressor(
        max_depth=3,
        random_state=42
    )
}

Блок 10 — CV evaluation on X_train only

In [10]:
results = []

for model_name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    cv_results = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=5,
        scoring=scoring,
        return_train_score=False
    )
    
    results.append({
        "model": model_name,
        "MAE_mean": -cv_results["test_mae"].mean(),
        "MAE_std": cv_results["test_mae"].std(),
        "RMSE_mean": -cv_results["test_rmse"].mean(),
        "RMSE_std": cv_results["test_rmse"].std(),
        "R2_mean": cv_results["test_r2"].mean(),
        "R2_std": cv_results["test_r2"].std()
    })

results_df = pd.DataFrame(results).sort_values("RMSE_mean")
display(results_df)

,model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
3,Ridge,18758.161542,1121.088500,33857.889344,8060.243798,0.802895,0.073313
2,LinearRegression,19791.303140,1464.107683,43613.528886,16218.650620,0.654856,0.224416
4,DecisionTreeRegressor_max_depth_3,33012.263973,2142.582715,48997.246238,6126.575745,0.592987,0.064216
0,DummyRegressor_mean,56340.483620,2891.420120,77051.137457,5826.958206,-0.003663,0.003976
1,DummyRegressor_median,54617.145145,2486.630126,78846.692345,5357.140765,-0.052599,0.033431


Блок 11 — baseline conclusions

## Stage 3 baseline conclusions

### Setup
- Target: SalePrice.
- X was created by dropping SalePrice and Id.
- Official Kaggle test.csv was not used.
- Local train/test split was created from train.csv only.
- Local test set was not evaluated.
- Cross-validation was performed only on X_train.

### Preprocessing
- Numeric features: SimpleImputer(strategy="median") + StandardScaler.
- Categorical features: SimpleImputer(strategy="constant", fill_value="None") + OneHotEncoder(handle_unknown="ignore").
- All preprocessing was placed inside Pipeline / ColumnTransformer.

### Models compared
- DummyRegressor(strategy="mean")
- DummyRegressor(strategy="median")
- LinearRegression
- Ridge
- DecisionTreeRegressor(max_depth=3)

### Interpretation
- DummyRegressor establishes a naive baseline.
- LinearRegression and Ridge test whether a simple linear model with one-hot encoded categories is already strong.
- Shallow DecisionTree tests a simple non-linear baseline without tuning.
- Best simple baseline is selected by CV RMSE / MAE on X_train only.

### Leakage checks
- No preprocessing was fitted outside Pipeline.
- No model was evaluated on local X_test.
- Official Kaggle test.csv was not used.
- Id was excluded from X.
- No outlier removal was performed.
- No hyperparameter tuning was performed.
- No log target transformation was used in the main baseline.